# Study 1 follow-up (post hoc, CPU only): paired contrasts and VinDr missingness checks

Reads the saved scores from the two runs of 3 October 2026 (Amendment 1 NIH notebook and Amendment 2 VinBigData notebook).
No images, no GPU, no internet. These analyses were defined after seeing the amendment results and are reported as post hoc.

**Inputs:** Add Input, then Your Work (Notebooks), add the Amendment 1 notebook and the VinBigData notebook. Accelerator **None**. Then Save & Run All.

In [ ]:
import os, json, time, numpy as np, pandas as pd
T0 = time.time(); SEED = 2026
def find(name, must=None):
    hits = []
    for root, dirs, files in os.walk("/kaggle/input"):
        if name in files and (must is None or must in root): hits.append(os.path.join(root, name))
    return sorted(hits)
npz = find("scores_all_templates.npz"); assert npz, "Amendment 1 output (scores_all_templates.npz) not found: add that notebook as input"
nih_dir = os.path.dirname(npz[0]); vin_s = find("scores_chexzero.npy", "vin_out")
assert vin_s, "VinBigData output (vin_out) not found: add that notebook as input"
vin_dir = os.path.dirname(vin_s[0]); print("NIH:", nih_dir, "\nVin:", vin_dir)
FINDINGS = ["Atelectasis","Cardiomegaly","Effusion","Infiltration","Mass","Nodule","Pneumonia",
            "Pneumothorax","Consolidation","Edema","Emphysema","Fibrosis","Pleural_Thickening","Hernia"]
class Cell:
    def __init__(self, s, y, pidx):
        o = np.argsort(s, kind="mergesort"); s = s[o]; self.y, self.p = y[o].astype(np.float64), pidx[o]
        self.starts = np.flatnonzero(np.r_[True, s[1:] != s[:-1]])
    def auc(self, w=None):
        wi = 1.0 if w is None else w[self.p]
        bp = np.add.reduceat(wi * self.y, self.starts); bn = np.add.reduceat(wi * (1 - self.y), self.starts)
        P, N = bp.sum(), bn.sum()
        return np.nan if P == 0 or N == 0 else float((bp * (np.cumsum(bn) - bn + 0.5 * bn)).sum() / (P * N))
def ci(a, lvl=0.95):
    a = a[~np.isnan(a)]
    return (float("nan"), float("nan")) if a.size == 0 else (float(np.quantile(a, (1-lvl)/2)), float(np.quantile(a, 1-(1-lvl)/2)))
def table(s, Y, mask, pidx, WB):
    idx = np.flatnonzero(mask); pt, bt = [], []
    for j in range(Y.shape[1]):
        c = Cell(s[idx, j], Y[idx, j], pidx[idx]); pt.append(c.auc()); bt.append([c.auc(w) for w in WB])
    return np.array(pt), np.array(bt).T
def mac(tab, E): pt, bt = tab; return float(np.nanmean(pt[E])), np.nanmean(bt[:, E], axis=1)
def fmt(p, c): return f"{p:+.3f} ({c[0]:+.3f} to {c[1]:+.3f})"
OUT = {}

In [ ]:
# NIH: paired contrasts across templates (patient bootstrap, 1,000, seed 2026)
S = np.load(npz[0]); coh = pd.read_csv(os.path.join(nih_dir, "cohort.csv"))
assert len(coh) == S["chexzero__T1"].shape[0], (len(coh), S["chexzero__T1"].shape)
te = (coh.set == "test").values; T = coh[te].reset_index(drop=True); Y = T[FINDINGS].values
_, pidx = np.unique(T.patient.values, return_inverse=True); NP = pidx.max() + 1
rng = np.random.default_rng(SEED); WB = [np.bincount(rng.integers(0, NP, NP), minlength=NP).astype(np.float64) for _ in range(1000)]
allm = np.ones(len(T), bool); E = list(range(14))
TAB = {k: table(S[k][te].astype(np.float64), Y, allm, pidx, WB) for k in S.files}
check = {k: round(mac(TAB[k], E)[0], 3) for k in ["chexzero__T1", "clear_concept__T1", "clear_direct__T1"]}
print("reproduction check (expect 0.695 / 0.635 / 0.700):", check)
nih = {}
for t in ["T1", "T2", "T3", "T4"]:
    pc, bc = mac(TAB[f"chexzero__{t}"], E)
    for m in ["clear_concept", "clear_direct"]:
        pm, bm = mac(TAB[f"{m}__{t}"], E); nih[f"{m}_minus_chexzero_{t}"] = (pm - pc, ci(bm - bc))
    pk, bk = mac(TAB[f"clear_concept__{t}"], E); pd_, bd = mac(TAB[f"clear_direct__{t}"], E)
    nih[f"concept_minus_direct_{t}"] = (pk - pd_, ci(bk - bd))
for m in ["chexzero", "clear_concept", "clear_direct"]:
    p1, b1 = mac(TAB[f"{m}__T1"], E)
    for t in ["T2", "T3", "T4"]:
        pt_, bt_ = mac(TAB[f"{m}__{t}"], E); nih[f"{m}_{t}_minus_T1"] = (pt_ - p1, ci(bt_ - b1))
OUT["nih"] = nih

In [ ]:
# VinDr: concept vs direct, missingness (site proxy), per-finding subgroup gaps (image bootstrap, 1,000, seed 2026)
vc = pd.read_csv(os.path.join(vin_dir, "cohort.csv"))
VF = [c for c in vc.columns if c not in ("image_id", "sex", "age", "age_band4") and not c.startswith("any_")]
VS = {k: np.load(os.path.join(vin_dir, f"scores_{k}.npy")).astype(np.float64) for k in ["chexzero", "clear_concept", "clear_direct"]}
assert len(vc) == len(VS["chexzero"]); print("VinDr findings:", VF)
Yv = vc[VF].values; n = len(vc); iidx = np.arange(n)
rng = np.random.default_rng(SEED); WV = [np.bincount(rng.integers(0, n, n), minlength=n).astype(np.float64) for _ in range(1000)]
vin = {}
allv = np.ones(n, bool); Ev = [j for j in range(len(VF)) if Yv[:, j].sum() >= 30]
TV = {k: table(VS[k], Yv, allv, iidx, WV) for k in VS}
pk, bk = mac(TV["clear_concept"], Ev); pd_, bd = mac(TV["clear_direct"], Ev); vin["concept_minus_direct"] = (pk - pd_, ci(bk - bd))
sex_known = vc.sex.isin(["M", "F"]).values; age_known = vc.age.notna().values
groups = {"sex_known": sex_known, "sex_missing": ~sex_known, "age_known": age_known, "age_missing": ~age_known}
for a in ["18-40", "40-60", "60+"]: groups[f"age_{a}"] = (vc.age_band4 == a).values
groups["sex_F"] = (vc.sex == "F").values; groups["sex_M"] = (vc.sex == "M").values
vin["n"] = {g: int(m.sum()) for g, m in groups.items()}
vin["prevalence_pct"] = {g: {VF[j]: round(100 * Yv[m, j].mean(), 1) for j in range(len(VF))} for g, m in groups.items()}
def elig(gs): return [j for j in range(len(VF)) if all(Yv[groups[g], j].sum() >= 30 for g in gs)]
TG = {(k, g): table(VS[k], Yv, m, iidx, WV) for k in VS for g, m in groups.items()}
for a, b in [("sex_known", "sex_missing"), ("age_known", "age_missing")]:
    Eg = elig([a, b]); vin[f"{a}_minus_{b}_findings"] = [VF[j] for j in Eg]
    for k in VS:
        pa, ba = mac(TG[(k, a)], Eg); pb, bb = mac(TG[(k, b)], Eg); vin[f"{k}:{a}_minus_{b}"] = (pa - pb, ci(ba - bb))
Es = elig(["sex_F", "sex_M"]); Ea = elig(["age_18-40", "age_40-60", "age_60+"])
vin["per_finding_sex_F_minus_M"] = {k: {VF[j]: round(float(TG[(k, "sex_F")][0][j] - TG[(k, "sex_M")][0][j]), 3) for j in Es} for k in VS}
vin["per_finding_age_18-40_minus_40-60"] = {k: {VF[j]: round(float(TG[(k, "age_18-40")][0][j] - TG[(k, "age_40-60")][0][j]), 3) for j in Ea} for k in VS}
# case-mix check: age gap recomputed with each finding left out in turn (CheXzero, CLEAR concept)
loo = {}
for k in ["chexzero", "clear_concept"]:
    loo[k] = {}
    for drop in Ea:
        E2 = [j for j in Ea if j != drop]; p1, b1 = mac(TG[(k, "age_18-40")], E2); p0, b0 = mac(TG[(k, "age_40-60")], E2)
        loo[k][VF[drop]] = (round(float(p1 - p0), 3), [round(float(x), 3) for x in ci(b1 - b0, 1 - 0.05/3)])
vin["age_gap_leave_one_finding_out"] = loo
OUT["vin"] = vin

In [ ]:
def jsonable(o):
    if isinstance(o, dict): return {str(k): jsonable(v) for k, v in o.items()}
    if isinstance(o, (list, tuple)): return [jsonable(x) for x in o]
    if isinstance(o, (np.floating, np.integer)): return o.item()
    return o
os.makedirs("/kaggle/working/followup_out", exist_ok=True)
json.dump(jsonable(OUT), open("/kaggle/working/followup_out/followup_results.json", "w"), indent=1)
print("===== FOLLOWUP SUMMARY =====")
print("NIH reproduction check:", check)
print("-- NIH paired contrasts, macro over 14 findings, 95% patient bootstrap")
for k, (p, c) in OUT["nih"].items(): print(f"  {k:<34} {fmt(p, c)}")
v = OUT["vin"]
print("-- VinDr concept minus direct:", fmt(*v["concept_minus_direct"]))
print("-- VinDr n:", v["n"])
for a, b in [("sex_known", "sex_missing"), ("age_known", "age_missing")]:
    print(f"-- VinDr {a} minus {b} ({len(v[a + '_minus_' + b + '_findings'])} findings):",
          " | ".join(f"{k} {fmt(*v[f'{k}:{a}_minus_{b}'])}" for k in VS))
print("-- VinDr prevalence % (known / missing sex; 18-40 / 40-60 / 60+):")
for f in VF:
    pv = v["prevalence_pct"]
    print(f"  {f:<20} {pv['sex_known'][f]:>5} / {pv['sex_missing'][f]:<5} | {pv['age_18-40'][f]:>5} / {pv['age_40-60'][f]:>5} / {pv['age_60+'][f]:<5}")
print("-- VinDr per-finding F minus M:", v["per_finding_sex_F_minus_M"])
print("-- VinDr per-finding 18-40 minus 40-60:", v["per_finding_age_18-40_minus_40-60"])
print("-- VinDr 18-40 minus 40-60, leaving one finding out (98.33% CI):", v["age_gap_leave_one_finding_out"])
print("minutes:", round((time.time() - T0) / 60, 1))
print("===== END =====")